# Flood Risk Prediction — SMOTE and LIME
### Manuela Munoz Ramirez

Two things here. First I try SMOTE on the tabular models to see if balancing the classes
does any better than the `class_weight="balanced"` we already use. Then LIME on the
Logistic Regression to see why it flags a given day.

Uses `common.py` like the other notebooks so the data and split match.

## Setup

In [ ]:
# imbalanced-learn and lime aren't in the env, install them
!pip install imbalanced-learn lime -q

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

import common

## 1. Load data and split

In [ ]:
df = common.load_data()
X, y = common.build_features(df)
X_train, X_test, y_train, y_test = common.chronological_split(X, y)

print("Train:", len(X_train), "| Test:", len(X_test))
print("Class balance in train:", dict(y_train.value_counts()))

## 2. SMOTE on the training set

Only the training set. If I resample the test set I'd be scoring on days that never
happened.

In [ ]:
sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)

print("Before:", dict(y_train.value_counts()))
print("After: ", dict(y_train_smote.value_counts()))

## 3. Train both ways and compare

class_weight vs SMOTE for LR and RF, scored on the same test set, with the baseline for
reference.

In [ ]:
results = []

lr_cw = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight="balanced"))
lr_cw.fit(X_train, y_train)
results.append(common.evaluate("LR (class_weight)", y_test,
                               lr_cw.predict(X_test), lr_cw.predict_proba(X_test)[:, 1]))

lr_sm = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr_sm.fit(X_train_smote, y_train_smote)
results.append(common.evaluate("LR (SMOTE)", y_test,
                               lr_sm.predict(X_test), lr_sm.predict_proba(X_test)[:, 1]))

rf_cw = RandomForestClassifier(n_estimators=400, max_depth=10,
                               class_weight="balanced", random_state=42, n_jobs=-1)
rf_cw.fit(X_train, y_train)
results.append(common.evaluate("RF (class_weight)", y_test,
                               rf_cw.predict(X_test), rf_cw.predict_proba(X_test)[:, 1]))

rf_sm = RandomForestClassifier(n_estimators=400, max_depth=10,
                               random_state=42, n_jobs=-1)
rf_sm.fit(X_train_smote, y_train_smote)
results.append(common.evaluate("RF (SMOTE)", y_test,
                               rf_sm.predict(X_test), rf_sm.predict_proba(X_test)[:, 1]))

results.append(common.persistence_baseline(df, y_test))

common.comparison_table(results)

## 4. What this shows

SMOTE comes out about the same as class_weight for LR, and a bit worse for RF. Makes
sense since class_weight already deals with the imbalance, so oversampling on top just
adds synthetic points without new information.

So we keep class_weight. Didn't run SMOTE on the LSTM because interpolating between time
windows doesn't give you a real hydrograph.

## 5. LIME on the Logistic Regression

SHAP (elsewhere in the repo) is the global view. LIME is per-day: for this one day, what
made the model call it high risk. That's closer to what someone reading a single alert
would ask.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(
    X_train.values,
    feature_names=list(X.columns),
    class_names=["Low risk", "High risk"],
    mode="classification",
    random_state=42,
)

# grab a real high-risk day from the test set
high_risk_idx = np.where(y_test.values == 1)[0][0]

explanation = explainer.explain_instance(
    X_test.values[high_risk_idx], lr_cw.predict_proba, num_features=4
)

print("High-risk day. Predicted prob:",
      round(lr_cw.predict_proba(X_test.values[high_risk_idx:high_risk_idx+1])[0][1], 3))
print()
for feat, weight in explanation.as_list():
    print(f"  {feat:30s} {weight:+.3f}")

### Plot it

In [ ]:
pairs = explanation.as_list()
labels = [p[0] for p in pairs]
weights = [p[1] for p in pairs]
colors = ["#2a9d5c" if w > 0 else "#9a9a9a" for w in weights]

plt.figure(figsize=(8, 3))
plt.barh(labels, weights, color=colors)
plt.axvline(0, color="black", linewidth=0.6)
plt.title("LIME - why this day was flagged (LR)")
plt.xlabel("contribution")
plt.tight_layout()
plt.show()

### A calm day too

Just to check it reasons the other way when the river is low.

In [ ]:
low_risk_idx = np.where(y_test.values == 0)[0][0]

exp_low = explainer.explain_instance(
    X_test.values[low_risk_idx], lr_cw.predict_proba, num_features=4
)

print("Low-risk day. Predicted prob:",
      round(lr_cw.predict_proba(X_test.values[low_risk_idx:low_risk_idx+1])[0][1], 3))
print()
for feat, weight in exp_low.as_list():
    print(f"  {feat:30s} {weight:+.3f}")

## Notes for the report

SMOTE: training set only, on LR and RF. No better than class_weight, a bit worse on RF,
so we kept class_weight. Not used on the LSTM.

LIME: per-day explanations on the LR. High-risk days are driven by yesterday's level and
the 7-day average, which is what you'd expect. Pairs with the SHAP global importances
already in the repo.